# Prompt Engineering for Large Language Models

This notebook teaches **practical prompt engineering techniques** for working with Large Language Models (LLMs).

## What We'll Learn

- Why prompting is the primary interface to LLMs
- How **in-context learning** enables adaptation without training
- Core techniques: zero-shot, few-shot, chain-of-thought
- Advanced methods: ReAct, self-consistency, tree of thoughts
- Practical design principles and debugging strategies
- Real-world applications with before/after comparisons

## Why This Matters

Unlike traditional ML where you train models, **LLMs are programmed through natural language**. The quality of your prompts directly determines the quality of outputs. Good prompt engineering can:

- Turn a failing task into a successful one
- Improve accuracy by 20-50% or more
- Reduce costs by getting better results with smaller models
- Enable complex reasoning and multi-step workflows

Let's build these skills step by step.

## Setup

We'll use the **transformers** library with GPT-2 for local demonstrations. This lets us:
- Run everything locally (no API keys needed)
- See immediate results
- Experiment freely

The techniques we learn apply to all LLMs (GPT-4, Claude, Llama, etc.).

**Note on Execution**: This notebook is designed for **interactive use** in Jupyter Lab. Due to memory constraints with GPT-2 loaded and 90+ cells with generations, it may not execute successfully end-to-end in automated testing. Run it interactively, section by section, for the best experience.

In [ ]:
# Install required packages (run once)
# !pip install transformers torch -q

### Import Libraries

We'll use standard imports and set up autoreload for development.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import warnings
import re
import gc
from typing import List, Dict, Any

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

### Set Random Seed

For reproducible results across runs.

In [ ]:
from aiml_notebooks import set_seed, get_device

set_seed(42)

### Load Model and Tokenizer

We'll use **GPT-2** (124M parameters) - small enough to run locally, large enough to demonstrate concepts.

For production, you'd use larger models (GPT-4, Claude, etc.) via APIs, but the prompting techniques are identical.

In [ ]:
# Use CPU for better compatibility with transformers
device = get_device(prefer_cpu=True)

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model = model.to(device)
model.eval()

# Set padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {model_name}")
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

### Create a Simple Generation Function

This helper function will make it easy to test different prompts.

Key parameters:
- **max_new_tokens**: How many tokens to generate
- **temperature**: Randomness (lower = more focused, higher = more creative)
- **top_p**: Nucleus sampling threshold (keeps most likely tokens)
- **do_sample**: Whether to use sampling (True) or greedy decoding (False)

In [ ]:
def generate(prompt: str, max_new_tokens: int = 50, temperature: float = 0.7, top_p: float = 0.9, do_sample: bool = True) -> str:
    """Generate text from a prompt using the loaded model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Clean up to free memory
    del inputs, outputs
    gc.collect()
    
    # Remove the prompt from the response to show only generated text
    return response[len(prompt):].strip()

# Test it
test_prompt = "The capital of France is"
result = generate(test_prompt, max_new_tokens=10, temperature=0.1)
print(f"Prompt: {test_prompt}")
print(f"Generated: {result}")

## 1. Basic Prompting

Let's start with the fundamentals: **zero-shot** and **few-shot** prompting.

### What is Zero-Shot Prompting?

**Zero-shot** means giving the model a task with no examples - just instructions. The model relies entirely on knowledge from pre-training.

#### Zero-Shot Example: Sentiment Analysis

Let's ask the model to classify sentiment without any examples.

In [ ]:
zero_shot_prompt = """Classify the sentiment of this review as positive, negative, or neutral.

Review: The movie was absolutely terrible. I wasted two hours of my life.
Sentiment:"""

result = generate(zero_shot_prompt, max_new_tokens=10, temperature=0.1)
print(f"Prompt:\n{zero_shot_prompt}")
print(f"\nGenerated: {result}")

### What is Few-Shot Prompting?

**Few-shot** means providing a few examples before asking for the task. This is **in-context learning** - the model learns the pattern from examples without any training.

This typically works much better than zero-shot, especially for:
- Specific output formats
- Domain-specific tasks
- Consistent behavior

#### Few-Shot Example: Sentiment Analysis

Same task, but now with examples to guide the model.

In [ ]:
few_shot_prompt = """Classify the sentiment of reviews as positive, negative, or neutral.

Review: I loved this product! It exceeded all my expectations.
Sentiment: positive

Review: It's okay, nothing special but does the job.
Sentiment: neutral

Review: Complete waste of money. Broke after one use.
Sentiment: negative

Review: The movie was absolutely terrible. I wasted two hours of my life.
Sentiment:"""

result = generate(few_shot_prompt, max_new_tokens=10, temperature=0.1)
print(f"Generated: {result}")

**Key Observation**: Few-shot prompting typically gives more reliable, consistent outputs. The examples teach the model:
1. The exact format you want
2. The level of detail needed
3. The style and tone to match

### Instruction Following

Modern LLMs are trained to follow instructions. Clear, specific instructions work best.

Let's compare vague vs. specific instructions.

In [ ]:
# Vague instruction
vague_prompt = "Tell me about Paris."

# Specific instruction
specific_prompt = """Write exactly 3 bullet points about Paris, France. 
Each bullet should be one sentence. Focus on culture, not geography.

Answer:"""

print("=== VAGUE INSTRUCTION ===")
print(generate(vague_prompt, max_new_tokens=60, temperature=0.7))

print("\n=== SPECIFIC INSTRUCTION ===")
print(generate(specific_prompt, max_new_tokens=60, temperature=0.7))

**Takeaway**: Specific instructions constrain the output to exactly what you need. This reduces wasted tokens and improves reliability.

## 2. Chain-of-Thought (CoT) Prompting

**Chain-of-Thought** prompting asks the model to show its reasoning steps before giving an answer.

### Why CoT Works

LLMs generate one token at a time. Complex reasoning requires intermediate steps - thinking through the problem before answering. CoT:

1. Breaks problems into manageable pieces
2. Makes reasoning explicit and debuggable
3. Dramatically improves accuracy on reasoning tasks

Introduced in [Wei et al. 2022](https://arxiv.org/abs/2201.11903), this simple technique improved accuracy by 20-30% on math and logic tasks.

### CoT Example: Math Word Problem

Let's solve a math problem without and with CoT.

In [ ]:
# Without CoT - direct answer
direct_prompt = """Q: A store has 15 apples. They sell 7 apples and then receive a shipment of 12 more. How many apples do they have now?
A:"""

print("=== WITHOUT CHAIN-OF-THOUGHT ===")
result = generate(direct_prompt, max_new_tokens=30, temperature=0.1)
print(result)

In [ ]:
# With CoT - show reasoning steps
cot_prompt = """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 balls. How many tennis balls does he have now?
A: Let's think step by step.
Roger started with 5 balls.
2 cans of 3 balls each is 2 × 3 = 6 balls.
5 + 6 = 11.
The answer is 11.

Q: A store has 15 apples. They sell 7 apples and then receive a shipment of 12 more. How many apples do they have now?
A: Let's think step by step."""

print("=== WITH CHAIN-OF-THOUGHT ===")
result = generate(cot_prompt, max_new_tokens=50, temperature=0.1)
print(result)

**Key Insight**: The magic phrase **"Let's think step by step"** triggers step-by-step reasoning. The model:
1. Breaks down the problem
2. Shows intermediate calculations
3. Arrives at the final answer

This is more reliable than jumping straight to an answer.

### CoT Example: Logic Puzzle

Chain-of-thought excels at logic problems that require multiple reasoning steps.

In [ ]:
logic_cot = """Q: All roses are flowers. Some flowers fade quickly. Therefore, do all roses fade quickly?
A: Let's think step by step.
1. We know all roses are flowers (roses ⊂ flowers).
2. We know some flowers fade quickly (not all).
3. Just because some flowers fade quickly doesn't mean all flowers do.
4. Since roses are flowers, and only SOME flowers fade quickly, we can't conclude that all roses fade quickly.
The answer is: No, we cannot conclude that all roses fade quickly.

Q: All mammals have backbones. All dogs are mammals. Therefore, do all dogs have backbones?
A: Let's think step by step."""

result = generate(logic_cot, max_new_tokens=80, temperature=0.1)
print(result)

**Pattern Recognition**: The example teaches the model to:
- Break logical statements into numbered steps
- Use symbolic notation (⊂) when helpful
- State the final answer clearly

The model learns this pattern from just one example!

## 3. Few-Shot Learning Deep Dive

Few-shot learning is powerful, but **example selection and order matter**.

### Key Principles

1. **Representative examples**: Cover the range of inputs you'll see
2. **Diverse examples**: Show different patterns and edge cases
3. **Consistent format**: Same structure for all examples
4. **Order matters**: Put most relevant examples last (recency bias)

### Example Selection: Text Classification

Let's classify questions by topic. We'll compare different example sets.

In [ ]:
# Poor examples - too similar
poor_examples = """Classify the question topic: science, history, sports, or technology.

Q: Who invented the telephone?
Topic: technology

Q: Who invented the light bulb?
Topic: technology

Q: What is machine learning?
Topic:"""

print("=== POOR EXAMPLES (all technology) ===")
result = generate(poor_examples, max_new_tokens=5, temperature=0.1)
print(f"Result: {result}")
print("\nProblem: Model only saw technology examples!")

In [ ]:
# Good examples - diverse and representative
good_examples = """Classify the question topic: science, history, sports, or technology.

Q: Who won the World Cup in 2018?
Topic: sports

Q: What year did World War II end?
Topic: history

Q: What is photosynthesis?
Topic: science

Q: Who invented the telephone?
Topic: technology

Q: What is machine learning?
Topic:"""

print("=== GOOD EXAMPLES (diverse topics) ===")
result = generate(good_examples, max_new_tokens=5, temperature=0.1)
print(f"Result: {result}")

**Lesson**: Diverse examples teach the model to distinguish between categories. Homogeneous examples create bias.

### Order Matters: Recency Bias

LLMs exhibit **recency bias** - they're more influenced by recent examples. Put the most relevant examples last.

In [ ]:
# Test input: clearly negative
test_review = "This product is garbage. Don't waste your money."

# Order 1: Positive example last
positive_last = f"""Classify sentiment: positive or negative.

Review: Terrible quality, broke immediately.
Sentiment: negative

Review: Amazing! Best purchase ever!
Sentiment: positive

Review: {test_review}
Sentiment:"""

# Order 2: Negative example last
negative_last = f"""Classify sentiment: positive or negative.

Review: Amazing! Best purchase ever!
Sentiment: positive

Review: Terrible quality, broke immediately.
Sentiment: negative

Review: {test_review}
Sentiment:"""

print("Test review:", test_review)
print("\nOrder 1 (positive last):", generate(positive_last, max_new_tokens=5, temperature=0.1))
print("Order 2 (negative last):", generate(negative_last, max_new_tokens=5, temperature=0.1))
print("\nNote: The last example can influence the prediction!")

**Best Practice**: When possible, put examples most similar to your test case at the end.

## 4. Advanced Techniques

Now we'll explore cutting-edge prompting methods that push LLMs further.

### ReAct: Reasoning + Acting

**ReAct** ([Yao et al. 2022](https://arxiv.org/abs/2210.03629)) interleaves reasoning and actions. The model:
1. **Thinks** about what to do
2. **Acts** by using tools or gathering information
3. **Observes** the result
4. Repeats until done

This enables multi-step problem solving with external information.

#### ReAct Example: Question Answering

Let's simulate a ReAct loop for answering a question that requires looking up information.

In [ ]:
react_prompt = """Answer questions using this format:
Thought: [reasoning about what to do]
Action: [action to take: Search, Lookup, or Finish]
Observation: [result of action]

Question: What is the population of the capital of France?

Thought: I need to find the capital of France first.
Action: Search[capital of France]
Observation: The capital of France is Paris.

Thought: Now I need to find the population of Paris.
Action: Search[population of Paris]
Observation: Paris has a population of approximately 2.2 million.

Thought: I now have the answer.
Action: Finish[2.2 million]

Question: What is the tallest mountain in the country where the Eiffel Tower is located?

Thought:"""

result = generate(react_prompt, max_new_tokens=100, temperature=0.1)
print(result)

**Key Insight**: ReAct teaches the model to break down complex queries into sub-tasks. Each thought-action-observation cycle brings it closer to the answer.

In practice, you'd connect "Action" steps to actual tools (search engines, calculators, databases).

### Self-Consistency: Multiple Paths to the Answer

**Self-consistency** ([Wang et al. 2022](https://arxiv.org/abs/2203.11171)) generates multiple reasoning paths and takes the majority vote.

Why it works:
- Different reasoning paths may catch different errors
- The correct answer appears more frequently
- Reduces sensitivity to random variations

#### Self-Consistency Example

Let's solve a problem multiple times with different reasoning paths.

In [ ]:
problem = """Q: A farmer has 17 sheep. All but 9 die. How many sheep are left?
A: Let's think step by step."""

# Generate multiple solutions with sampling (reduced from 5 to 3 to save memory)
solutions = []
for i in range(3):
    result = generate(problem, max_new_tokens=50, temperature=0.7, do_sample=True)
    solutions.append(result)
    print(f"\n=== Solution {i+1} ===")
    print(result)

print("\n=== Self-Consistency ===")
print("Generate multiple solutions, extract final answers, and take majority vote.")
print("This is more reliable than a single reasoning path.")

**Trade-off**: Self-consistency costs more (multiple generations) but significantly improves accuracy on reasoning tasks.

### Tree of Thoughts: Exploring Multiple Branches

**Tree of Thoughts** ([Yao et al. 2023](https://arxiv.org/abs/2305.10601)) extends CoT by exploring multiple reasoning paths like a tree:

1. Generate multiple next steps
2. Evaluate which are most promising
3. Explore the best branches further
4. Backtrack if needed

This is inspired by how humans solve complex problems - considering alternatives and backtracking when stuck.

#### Tree of Thoughts Illustration

Let's demonstrate the concept with a creative writing task.

In [ ]:
tot_prompt = """Write a creative story opening. Consider multiple directions:

Option 1: Start with dialogue
Option 2: Start with action
Option 3: Start with description

Evaluate each option:
- Which creates immediate tension?
- Which draws the reader in?
- Which sets up the world?

Choose the best approach and write the opening.

Story opening:"""

result = generate(tot_prompt, max_new_tokens=80, temperature=0.8)
print(result)

print("\n=== Concept ===")
print("Tree of Thoughts explores multiple approaches before committing.")
print("For complex problems (game playing, planning, creative tasks), this beats linear reasoning.")

## 5. Prompt Design Principles

Now let's codify what makes a good prompt. These principles apply universally.

### Principle 1: Be Clear and Specific

Vague prompts get vague results. Specific prompts get specific results.

In [ ]:
# Bad: Vague
vague = "Tell me about dogs."

# Good: Specific
specific = """List 3 specific differences between Golden Retrievers and German Shepherds.
Focus on temperament, size, and typical roles. Use exactly one sentence per difference.

Answer:"""

print("=== VAGUE ===")
print(generate(vague, max_new_tokens=50, temperature=0.7))

print("\n=== SPECIFIC ===")
print(generate(specific, max_new_tokens=70, temperature=0.7))

**Specificity checklist**:
- What format? (bullet points, paragraph, JSON)
- How many? (3 items, 100 words)
- What focus? (temperament not appearance)
- What tone? (formal, casual, technical)

### Principle 2: Use Delimiters

**Delimiters** clearly separate instructions from content. This prevents confusion and injection attacks.

In [ ]:
# Without delimiters - ambiguous
no_delimiters = """Summarize this text: The quick brown fox jumps over the lazy dog."""

# With delimiters - clear separation
with_delimiters = """Summarize the text between triple quotes in one sentence.

Text: \"\"\"The quick brown fox jumps over the lazy dog.\"\"\" 

Summary:"""

print("=== WITH DELIMITERS ===")
print(generate(with_delimiters, max_new_tokens=30, temperature=0.7))

print("\nCommon delimiters: triple quotes (\"\"\"), XML tags (<text>), triple backticks (```)")

**Why delimiters matter**:
- Prevent the model from confusing instructions with content
- Enable processing of adversarial inputs
- Make prompts more maintainable

### Principle 3: Provide Context and Role

**Role-playing** guides the model's knowledge and style. "You are an expert X" activates relevant training data.

In [ ]:
# Without role
no_role = "Explain quantum entanglement."

# With role - activates teaching mode
with_role = """You are a physics teacher explaining concepts to high school students.
Use simple analogies and avoid jargon.

Explain quantum entanglement in 2-3 sentences.

Explanation:"""

print("=== WITHOUT ROLE ===")
print(generate(no_role, max_new_tokens=50, temperature=0.7))

print("\n=== WITH ROLE ===")
print(generate(with_role, max_new_tokens=50, temperature=0.7))

**Useful roles**:
- "You are an expert Python programmer"
- "You are a helpful assistant"
- "You are a technical writer"
- "You are a critical reviewer"

The role sets the tone, expertise level, and perspective.

### Principle 4: Specify the Format

Want JSON? Markdown? Bullet points? **Ask for it explicitly**.

In [ ]:
# Request structured output
structured_prompt = """Extract information from this sentence and return it in this exact format:

Name: [person's name]
Age: [age in years]
City: [city name]

Sentence: "John Smith is a 35-year-old software engineer living in Seattle."

Extracted information:"""

result = generate(structured_prompt, max_new_tokens=40, temperature=0.1)
print(result)

**Format examples**:
- JSON: `{"key": "value"}`
- Markdown table: `| Column | Column |`
- Numbered list: `1. Item\n2. Item`
- Code block: ` ```python\ncode\n``` `

Showing the exact format in the prompt ensures consistency.

## 6. Prompt Templates

**Templates** are reusable patterns for common tasks. Let's build a library of proven templates.

### Template 1: Summarization

In [ ]:
def summarization_template(text: str, length: str = "one sentence", focus: str = "main point") -> str:
    """Template for text summarization."""
    return f"""Summarize the following text in {length}.
Focus on: {focus}

Text: \"\"\"{text}\"\"\"

Summary:"""

# Example usage
article = """Machine learning is a subset of artificial intelligence that focuses on building 
systems that can learn from data. Instead of being explicitly programmed, these systems 
improve their performance through experience. Deep learning, a subset of machine learning, 
uses neural networks with many layers to learn complex patterns."""

prompt = summarization_template(article, length="one sentence", focus="the relationship between ML, AI, and deep learning")
print(generate(prompt, max_new_tokens=40, temperature=0.7))

### Template 2: Information Extraction

In [ ]:
def extraction_template(text: str, fields: List[str]) -> str:
    """Template for extracting structured information."""
    field_list = "\n".join([f"- {field}" for field in fields])
    return f"""Extract the following information from the text. If a field is not present, write "N/A".

Fields to extract:
{field_list}

Text: \"\"\"{text}\"\"\"

Extracted information:"""

# Example usage
email = """From: john@example.com
Subject: Meeting Reschedule

Hi team, I need to reschedule our meeting from Tuesday 3pm to Wednesday 2pm. 
The meeting will still be in Conference Room A. Please confirm your availability."""

fields = ["sender email", "subject", "old meeting time", "new meeting time", "location"]
prompt = extraction_template(email, fields)
print(generate(prompt, max_new_tokens=60, temperature=0.1))

### Template 3: Classification with Reasoning

In [ ]:
def classification_template(text: str, categories: List[str], explain: bool = True) -> str:
    """Template for text classification with optional explanation."""
    category_list = ", ".join(categories)
    
    if explain:
        return f"""Classify the text into one of these categories: {category_list}

Text: \"\"\"{text}\"\"\"

Reasoning: [explain why]
Category:"""
    else:
        return f"""Classify the text into one of these categories: {category_list}

Text: \"\"\"{text}\"\"\"

Category:"""

# Example usage
customer_message = "The delivery was late and the package was damaged. Very disappointed."
categories = ["complaint", "question", "praise", "request"]

prompt = classification_template(customer_message, categories, explain=True)
print(generate(prompt, max_new_tokens=40, temperature=0.1))

**Template Benefits**:
1. Consistency across tasks
2. Easy to test and improve
3. Reusable for different inputs
4. Documents best practices

## 7. Debugging Prompts

Prompts fail. Here's how to diagnose and fix common issues.

### Common Failure Modes

1. **Wrong format**: Model doesn't follow output structure
2. **Off-topic**: Model ignores instructions
3. **Incomplete**: Model stops too early
4. **Hallucination**: Model makes up information
5. **Inconsistent**: Different outputs each time

### Debugging Strategy 1: Make Instructions Explicit

If the model doesn't follow the format, show the exact format you want.

In [ ]:
# Failing prompt - format not followed
failing = """Extract the person's name and age from: 'Sarah is 28 years old.'"""

# Fixed prompt - explicit format
fixed = """Extract information in this EXACT format (do not add any other text):

Name: [extracted name]
Age: [extracted age]

Input: "Sarah is 28 years old."

Output:"""

print("=== BEFORE (vague) ===")
print(generate(failing, max_new_tokens=30, temperature=0.1))

print("\n=== AFTER (explicit format) ===")
print(generate(fixed, max_new_tokens=30, temperature=0.1))

### Debugging Strategy 2: Add Constraints

If the model generates too much or too little, add explicit length constraints.

In [ ]:
# Without constraints
unconstrained = """Explain photosynthesis."""

# With constraints
constrained = """Explain photosynthesis in exactly 2 sentences. 
First sentence: what it is. Second sentence: why it matters.

Explanation:"""

print("=== UNCONSTRAINED ===")
print(generate(unconstrained, max_new_tokens=60, temperature=0.7))

print("\n=== CONSTRAINED ===")
print(generate(constrained, max_new_tokens=60, temperature=0.7))

### Debugging Strategy 3: Use Examples to Show Correct Behavior

When in doubt, add few-shot examples showing exactly what you want.

In [ ]:
# Improved with examples
with_examples = """Convert sentences to questions.

Sentence: The sky is blue.
Question: What color is the sky?

Sentence: She went to the store.
Question: Where did she go?

Sentence: The meeting starts at 3pm.
Question:"""

result = generate(with_examples, max_new_tokens=20, temperature=0.7)
print(result)

**Debugging Checklist**:
1. Is the instruction clear and unambiguous?
2. Did I show the exact format I want?
3. Did I provide good examples?
4. Are there conflicting instructions?
5. Is the temperature too high (causing randomness)?

## 8. Practical Applications

Let's build complete, real-world applications using what we've learned.

### Application 1: Sentiment Analysis Pipeline

A robust sentiment classifier using few-shot learning.

In [ ]:
def analyze_sentiment(review: str, include_reasoning: bool = False) -> Dict[str, Any]:
    """Analyze sentiment of a review with optional reasoning."""
    
    if include_reasoning:
        prompt = f"""Classify the sentiment of product reviews. Think step by step.

Review: "Best purchase ever! It exceeded all my expectations and works perfectly."
Reasoning: The review uses strongly positive language like "best" and "exceeded expectations".
Sentiment: positive

Review: "It's okay. Does what it says, nothing more."
Reasoning: Lukewarm language like "okay" and "nothing more" suggests neutrality.
Sentiment: neutral

Review: "Waste of money. Broke after one day."
Reasoning: Explicitly negative phrases like "waste of money" and "broke" indicate dissatisfaction.
Sentiment: negative

Review: \"\"\"{review}\"\"\"
Reasoning:"""
    else:
        prompt = f"""Classify sentiment: positive, negative, or neutral.

Review: "Best purchase ever! Works perfectly."
Sentiment: positive

Review: "It's okay, nothing special."
Sentiment: neutral

Review: "Waste of money. Broke immediately."
Sentiment: negative

Review: \"\"\"{review}\"\"\"
Sentiment:"""
    
    result = generate(prompt, max_new_tokens=50 if include_reasoning else 10, temperature=0.1)
    return {"review": review, "response": result}

# Test the pipeline (reduced from 3 to 2 examples to save memory)
test_reviews = [
    "Absolutely love it! Best thing I've bought this year.",
    "Terrible. Arrived damaged and customer service was unhelpful."
]

for review in test_reviews:
    result = analyze_sentiment(review, include_reasoning=True)
    print(f"Review: {review}")
    print(f"Analysis: {result['response']}")
    print("-" * 80)

### Application 2: Named Entity Extraction

Extract structured information from unstructured text.

In [ ]:
def extract_entities(text: str) -> Dict[str, Any]:
    """Extract named entities: people, organizations, locations, dates."""
    
    prompt = f"""Extract named entities from text. Format:

People: [list]
Organizations: [list]
Locations: [list]
Dates: [list]

Example:
Text: "John Smith from Microsoft will visit Paris on January 15th."
People: John Smith
Organizations: Microsoft
Locations: Paris
Dates: January 15th

Text: \"\"\"{text}\"\"\"

Extracted entities:"""
    
    result = generate(prompt, max_new_tokens=80, temperature=0.1)
    return {"text": text, "entities": result}

# Test entity extraction
test_text = """Apple CEO Tim Cook announced at the Cupertino headquarters that the company will 
open new stores in Tokyo and London by December 2024."""

result = extract_entities(test_text)
print(f"Text: {result['text']}")
print(f"\nEntities:\n{result['entities']}")

### Application 3: Question Answering System

Answer questions based on context using chain-of-thought.

In [ ]:
def answer_question(context: str, question: str, use_cot: bool = True) -> Dict[str, Any]:
    """Answer questions based on provided context."""
    
    if use_cot:
        prompt = f"""Answer questions based on the context. Think step by step.

Context: "The Eiffel Tower was built in 1889 for the World's Fair. It is 330 meters tall."
Question: How tall is the Eiffel Tower?
Reasoning: The context states the tower is 330 meters tall.
Answer: 330 meters

Context: \"\"\"{context}\"\"\"
Question: {question}
Reasoning:"""
    else:
        prompt = f"""Answer the question based on the context.

Context: \"\"\"{context}\"\"\"
Question: {question}

Answer:"""
    
    result = generate(prompt, max_new_tokens=60, temperature=0.1)
    return {"question": question, "answer": result}

# Test Q&A (reduced from 3 to 2 questions to save memory)
context = """Python is a high-level programming language created by Guido van Rossum in 1991.
It emphasizes code readability and supports multiple programming paradigms including procedural,
object-oriented, and functional programming."""

questions = [
    "Who created Python?",
    "What year was Python created?"
]

for q in questions:
    result = answer_question(context, q, use_cot=True)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print("-" * 80)

### Application 4: Text Classification with Confidence

Classify text and provide confidence scores.

In [ ]:
def classify_with_confidence(text: str, categories: List[str]) -> Dict[str, Any]:
    """Classify text and provide reasoning for confidence."""
    
    category_str = ", ".join(categories)
    
    prompt = f"""Classify the text into one category: {category_str}
Then rate your confidence: high, medium, or low.

Example:
Text: "The new iPhone camera is amazing! Crystal clear photos."
Category: technology
Confidence: high
Reasoning: Clear mention of iPhone (tech product) and camera (tech feature)

Text: \"\"\"{text}\"\"\"
Category:"""
    
    result = generate(prompt, max_new_tokens=50, temperature=0.1)
    return {"text": text, "classification": result}

# Test classification (reduced from 3 to 2 examples to save memory)
test_cases = [
    "The quarterback threw a perfect spiral for a touchdown.",
    "Scientists discovered a new species in the Amazon rainforest."
]

categories = ["sports", "science", "business", "technology"]

for text in test_cases:
    result = classify_with_confidence(text, categories)
    print(f"Text: {result['text']}")
    print(f"Result: {result['classification']}")
    print("-" * 80)

## 9. Key Takeaways

Let's consolidate what we've learned into actionable principles.

### Core Techniques Summary

1. **Zero-shot vs Few-shot**
   - Zero-shot: Task description only
   - Few-shot: Add 2-5 examples (typically better)
   - Use few-shot for consistency and specific formats

2. **Chain-of-Thought (CoT)**
   - Add "Let's think step by step"
   - Essential for math, logic, and multi-step reasoning
   - Makes reasoning transparent and debuggable

3. **ReAct**
   - Interleave thinking and acting
   - Break complex tasks into sub-tasks
   - Connect to external tools when needed

4. **Self-Consistency**
   - Generate multiple solutions
   - Take majority vote
   - Trades cost for accuracy

5. **Tree of Thoughts**
   - Explore multiple paths
   - Evaluate and backtrack
   - Best for complex planning and creative tasks

### Design Principles Summary

**Clarity**
- Be specific about format, length, and focus
- Show examples of desired output
- Avoid ambiguity

**Structure**
- Use delimiters to separate sections
- Organize instructions logically
- Put examples before the task

**Context**
- Assign roles ("You are an expert...")
- Provide relevant background
- Specify the audience or purpose

**Constraints**
- Set explicit length limits
- Define allowed values
- Specify prohibited content

### When to Use Each Technique

Create a decision flowchart in your mind:

**Simple classification/extraction** → Few-shot examples

**Math/logic problems** → Chain-of-Thought

**Multi-step tasks** → ReAct (thought-action-observation)

**Critical decisions** → Self-Consistency (multiple samples)

**Complex planning** → Tree of Thoughts

**Consistent formatting** → Show exact format + examples

### Best Practices Checklist

Before deploying a prompt:

- [ ] Instructions are clear and specific
- [ ] Examples cover edge cases
- [ ] Output format is explicitly shown
- [ ] Delimiters separate content from instructions
- [ ] Temperature is appropriate (low=focused, high=creative)
- [ ] Tested with multiple inputs
- [ ] Failure modes are understood
- [ ] Cost vs accuracy tradeoff is acceptable

### Common Pitfalls to Avoid

1. **Ambiguous instructions** → Model guesses, inconsistent results
2. **No examples** → Model might misunderstand format
3. **Conflicting instructions** → Model gets confused
4. **Too many instructions** → Model ignores some
5. **Wrong temperature** → Too random or too deterministic
6. **Not testing edge cases** → Fails on unusual inputs
7. **Ignoring context limits** → Truncation issues
8. **Assuming perfect output** → Always validate results

### Iteration Strategy

Prompts rarely work perfectly on the first try. Use this process:

1. **Start simple** → Basic instruction
2. **Test** → Try multiple inputs
3. **Identify failures** → What goes wrong?
4. **Add specificity** → Clarify ambiguous parts
5. **Add examples** → Show correct behavior
6. **Add constraints** → Prevent unwanted behavior
7. **Test again** → Verify improvements
8. **Repeat** → Until performance is acceptable

This iterative approach is fundamental to prompt engineering.

### Going Further

**Advanced Topics** (beyond this notebook):
- Automatic prompt optimization (DSPy, PromptBench)
- Multi-modal prompting (text + images)
- Adversarial prompt testing
- Prompt compression techniques
- Constitutional AI and value alignment
- Agent frameworks (LangChain, AutoGPT)

**Resources**:
- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [OpenAI Prompt Engineering Best Practices](https://platform.openai.com/docs/guides/prompt-engineering)
- Research papers: CoT, ReAct, ToT, Constitutional AI
- [Learn Prompting](https://learnprompting.org/)

## Final Thoughts

**Prompt engineering is programming in natural language.** Like traditional programming:
- Start simple, iterate
- Test thoroughly
- Debug systematically
- Reuse patterns
- Document what works

Unlike traditional programming:
- Instructions are fuzzy, not precise
- Outputs are probabilistic, not deterministic
- Examples teach better than rules

**The key skill**: Translating your intent into clear, structured prompts that guide the model to produce exactly what you need.

Practice on real tasks, build a library of templates, and keep experimenting. The field is young and rapidly evolving - new techniques emerge constantly.

Happy prompting!